# Training GPT-2 From Scratch

This notebook is the complete code companion to the To Data & Beyond step-by-step tutorial. It preserves the tutorial's code order for loading OpenWebText through Deep Lake, configuring GPT-2, training with Hugging Face Trainer, and running inference.

## Before you run it

- The original tutorial pins `transformers==4.32.0`, `deeplake==3.6.19`, `wandb==0.15.8`, and `accelerate==0.22.0`. Use a compatible Python environment.
- Full pretraining was reported to require eight 40 GB NVIDIA A100 GPUs for roughly 40–45 hours. A standard laptop or free Colab runtime is not suitable for the full run.
- `wandb login` prompts for your own API key. Never store credentials in this notebook.
- Execution outputs are intentionally cleared. The notebook was checked structurally, not execution-tested.

## 1. Setting Up Working Environments

We will start with installing the packages we will work with in this article:

- Transformers : For working with transformer-based models like GPT-2.
- DeepLake : For managing large datasets.
- WandB : For experiment tracking.
- Accelerate : For optimizing and speeding up model training.

In [ ]:
!pip install -q transformers==4.32.0 deeplake==3.6.19 wandb==0.15.8 accelerate==0.22.0

Next, we will log in to Weight and Bias for the sake of reporting. You will need to have an account there and provide an API key.

In [ ]:
!wandb login

You will need to use an 8x NVIDIA A100 instance comprising 40GB of memory for around 40 hours to fully train the model with the

## 2. Load Dataset from Deep Lake

During the pre-training process, we will use the Activeloop datasets to stream the samples seamlessly, batch by batch. This approach proves beneficial for resource management as loading the entire dataset directly into memory is unnecessary.

Consequently, it greatly helps in optimizing resource usage. You can quickly load the dataset, and it automatically handles the streaming process without requiring any special configurations.

We will start by loading the openwebtext dataset , a collection of Reddit posts with at least three upvotes. This dataset is well-suited for acquiring broad knowledge to build a foundational model for general purposes.

The code below will instantiate a dataset object capable of retrieving the data points for both training and validation sets. Afterward, we can print the variable to examine the dataset’s characteristics.

In [ ]:
import deeplake

ds = deeplake.load('hub://activeloop/openwebtext-train')
ds_val = deeplake.load('hub://activeloop/openwebtext-val')

print(ds)
print(ds[0].text.text())

The returned data consists of two tensors: text containing the textual input and tokens representing the tokenized version of the content. We can also index through the dataset and access each column by using .text and convert the row to textual format by calling the .text() method.

The next step will be crafting a PyTorch Dataset class that leverages the loader object and ensures compatibility with the framework. The Dataset class handles both dataset formatting and any desired preprocessing steps to be applied. In this instance, our objective is to tokenize the samples. We will load the GPT-2 tokenizer model from the Transformers library to achieve this.

For this specific model, we need to set a padding token (which may not be required for other models), and for this specific purpose, we have chosen to utilize the end of sentence eos_token to set the loaded tokenizer’s pad_token method.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

Next, we will create dataloaders from the Deep Lake datasets. In doing so, we also specify a transform that tokenizes the texts of the dataset on the fly.

In [ ]:
# define transform to tokenize texts
def get_tokens_transform(tokenizer):
    def tokens_transform(sample_in):
        tokenized_text = tokenizer(
            sample_in["text"],
            truncation=True,
            max_length=512,
            padding='max_length',
            return_tensors="pt"
        )
        tokenized_text = tokenized_text["input_ids"][0]
        return {
            "input_ids": tokenized_text,
            "labels": tokenized_text
        }
    return tokens_transform

# create data loaders
ds_train_loader = ds.dataloader()\
    .batch(32)\
    .transform(get_tokens_transform(tokenizer))\
    .pytorch()

ds_eval_train_loader = ds_val.dataloader()\
    .batch(32)\
    .transform(get_tokens_transform(tokenizer))\
    .pytorch()

It is important to note that we have formatted the dataset so that each sample is comprised of two components: input_ids and labels. input_ids are the tokens the model will use as inputs, while labels are the tokens the model will try to predict.

Currently, both keys contain the same tokenized text. However, the trainer object from the Transformers library will automatically shift the labels by one token, preparing them for training.

## 3. Loading the Model & Tokenizer

We will use an existing publicly available implementation of the GPT-2 architecture. This approach allows us to scale the model quickly using available hyperparameters, including the number of layers, embedding dimension, and attention heads.

Additionally, we will capitalize on the success of established architectures while maintaining the flexibility to modify the model size to accommodate our available resources.

We will load the GPT-2 pre-trained model from the Huggingface hub; the approach presented here can be easily adapted to work with various architectures.

Initially, we examine the default hyperparameters by loading the configuration file and reviewing the choices made in the architecture design.

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("gpt2")
print(config)

We can see that we can have significant control over almost every aspect of the network by manipulating the configuration settings. However, we will focus on the following parameters:

- n_layer: This indicates the number of stacking decoder components and defines the embedding layer’s hidden dimension.
- n_positions and n_ctx: They represent the maximum number of input tokens.
- n_head: This is used to change the number of attention heads in each attention component.

You can read the documentation to gain a more comprehensive understanding of the remaining parameters. We will start by initializing the model using the default configuration and then count the number of parameters it contains, which will serve as a baseline.

To achieve this, we utilize the GPT2LMHeadModel class, which takes the config variable as input and then proceeds to loop through the parameters, summing them up accordingly.

In [ ]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f"GPT-2 size: {model_size/1e6:.1f}M parameters")

As shown, the GPT-2 model is relatively small (124M) when compared to the current state-of-the-art large language models. If you wanted to train a larger model, you could modify the architecture to scale it up slightly.

As we previously described the selected parameters, we can create a network with 32 layers and an embedding size of 1600. It is worth noting that if not specified, the hidden dimensionality of the linear layers will be 4 × n_embd.

In [ ]:
config.n_layer = 32
config.n_embd = 1600
config.n_positions = 512
config.n_ctx = 512
config.n_head = 32

Now, we proceed to load the model with the updated hyperparameters.

In [ ]:
model_1b = GPT2LMHeadModel(config)

model_size = sum(t.numel() for t in model_1b.parameters())
print(f"GPT2-1B size: {model_size/1e6:.1f}M parameters")

The modifications led to a model with 1 billion parameters. It is possible to scale the network further to be more in line with the newest state-of-the-art models, which often have more than 80 layers. However, let’s continue with this lesson’s 124M parameters model.

## 4. Training the Model

The final step in the process involves initializing the training loop. We utilize the Transformers library’s Trainer class, which takes the necessary parameters for training the model. However, before proceeding, we need to create a TrainingArguments object that defines all the essential arguments.

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="GPT2-training-scratch-openwebtext",
    evaluation_strategy="steps",
    save_strategy="steps",
    eval_steps=500,
    save_steps=500,
    num_train_epochs=2,
    logging_steps=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    weight_decay=0.1,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    bf16=True,
    ddp_find_unused_parameters=False,
    run_name="GPT2-scratch-openwebtext",
    report_to="wandb"
)

Note that we set the per_device_train_batch_size and the per_device_eval_batch_size variables to 1 as the batch size is already specified by the dataloader we created earlier.

There are over 90 parameters available for adjustment. Find a comprehensive list with explanations in the documentation . It is important to note that if there is an “out of memory” error while attempting to train, a smaller batch_size can be used.

Additionally, the bf16 flag, which trains the model using lower precision floating numbers, is only available on high-end GPU devices. If unavailable, it can be substituted with the argument fp16=True .

Notice also that we set the parameter report_to to wandb; that is, we are sending the training metrics to Weights and Biases so that we can see a real-time report of how the training is going. However, you need to provide wandb API key.

Next, we define the TrainerWithDataLoaders class, a subclass of Trainer where we override the get_train_dataloader and get_eval_dataloader methods to return our previously defined data loaders.

In [ ]:
from transformers import Trainer

class TrainerWithDataLoaders(Trainer):
    def __init__(self, *args, train_dataloader=None, eval_dataloader=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_dataloader = train_dataloader
        self.eval_dataloader = eval_dataloader

    def get_train_dataloader(self):
        return self.train_dataloader

    def get_eval_dataloader(self, dummy):
        return self.eval_dataloader

The process initiates with a call to the . train() method.

In [ ]:
trainer = TrainerWithDataLoaders(
    model=model,
    args=args,
    train_dataloader=ds_train_loader,
    eval_dataloader=ds_eval_train_loader,
)

trainer.train()

The Trainer object will handle model evaluation during training, as specified in the eval_steps argument, and save checkpoints based on the previously defined in save_steps . The model takes 45 hours of training on 8x NVIDIA A100. Here’s the training report on Weights and Biases. The following report shows that the training loss decreased relatively smoothly as iterations passed.

The original article includes a Weights & Biases training-loss report at this point. Run the training cells with your own W&B project to produce a fresh report.

## 5. Inference

Once the pre-training process is complete, we proceed with the inference stage to observe our model in action and evaluate its capabilities. As specified, the Trainer will store the intermediate checkpoints in a designated directory called ./GPT2 -training-scratch-openwebtext.

The most efficient approach to utilize the model involves leveraging the Transformers pipeline functionality, which automatically loads both the model and tokenizer, making them ready for text generation.

Below is the code snippet that establishes a pipeline object utilizing the pre-trained model alongside the tokenizer we defined in the preceding section. This pipeline enables text generation.

In [ ]:
from transformers import pipeline
pipe = pipeline("text-generation",
                model="./GPT2-scratch-openwebtext",
                tokenizer=tokenizer,
                device="cuda:0")

The pipeline object leverages the powerful Transformers .generate() method internally, offering exceptional flexibility in managing the text generation process.

We can use methods like min_length to define a minimum number of tokens to be generated, max_length to limit the newly generated tokens, temperature to control the generation process between randomness and most likely, and lastly, do_sample to modify the completion process, switching between a greedy approach that always selects the most probable token and other sampling methods, such as beam search or diverse search. We only set the num_return_sequences to limit the number of generated sequences.

In [ ]:
txt = "The house prices dropped down"

completion = pipe(txt, num_return_sequences=1)
print(completion)

The code will attempt to generate a completion for the given input sequence using the knowledge it has acquired from the training dataset. It aims to finish the following sequence: The house prices dropped down while being relevant and contextually appropriate.

Even with a brief training period, the model exhibits a good grasp of the language, generating grammatically correct and contextually coherent sentences.